# Practice #3. "Dynamic predictive models"

Stationarity, differencing, autocorrelation, AR forecasting and SSA.

Fill in the cells tagged `graded`, keeping every name and signature exactly as
given — they are graded automatically.

**Two series below, not interchangeable:**
- `demo_stationary`, `demo_random_walk` — short seeded synthetic series, used
  only to show what each test responds to.
- `series` — the real Melbourne temperatures. "Your series" always means this.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import acf, adfuller, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

def find_data_dir():
    """The repo's data/ directory, wherever the kernel happens to start."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data" / "airline-passengers.csv").exists():
            return folder / "data"
    raise FileNotFoundError("data/ not found - run this notebook inside the repo")


DATA_DIR = globals().get("DATA_DIR", find_data_dir())
RANDOM_SEED = 42

## 0. Data

In [ ]:
def load_series(path, time_col, value_col):
    """Read a CSV into a float Series named "y" on a DatetimeIndex."""
    # TODO: same contract as Practice 1 and 2.
    raise NotImplementedError

In [ ]:
series = load_series(DATA_DIR / "daily-min-temperatures.csv", "Date", "Temp")
print(f"{len(series)} points, {series.index.min():%Y-%m-%d} to {series.index.max():%Y-%m-%d}")

rng = np.random.default_rng(RANDOM_SEED)
demo_stationary = pd.Series(rng.normal(0, 1, 500))
demo_random_walk = pd.Series(rng.normal(0, 1, 500).cumsum())

fig, axes = plt.subplots(1, 2, figsize=(20, 4))
demo_stationary.plot(ax=axes[0], title="stationary: white noise")
demo_random_walk.plot(ax=axes[1], title="non-stationary: random walk");

## 1. Stationarity and the Dickey-Fuller test

A series is stationary when its mean, variance and autocovariance do not depend
on *when* you look. AR, MA and ARMA all assume it.

ADF's null is **"there is a unit root"** — non-stationary. So a **small**
p-value rejects the null and means *stationary*. Inverting this is the most
common error in this practice.

In [ ]:
def adf_test(series, significance=0.05):
    """Augmented Dickey-Fuller test.

    Returns {"statistic", "p_value", "used_lag", "n_obs", "critical_values"
    (dict), "is_stationary" (bool)}.

    ADF's null is "has a unit root" = non-stationary. Small p => stationary.
    """
    # TODO: unpack the tuple adfuller(series) returns into that dict.
    raise NotImplementedError

In [ ]:
for name, candidate in (("white noise", demo_stationary),
                        ("random walk", demo_random_walk),
                        ("temperatures", series)):
    result = adf_test(candidate)
    verdict = "stationary" if result["is_stationary"] else "NON-stationary"
    print(f"{name:>14}: p = {result['p_value']:.2e}  ->  {verdict}")

### 1.1 Differencing

Replacing `y[t]` with `y[t] - y[t-1]` removes a unit root. One difference
usually kills a linear trend, two handle a quadratic one. Over-differencing adds
noise, so stop as soon as the test agrees.

Practice 4 reuses this: the number of differences is the `d` in ARIMA(p, d, q).

In [ ]:
def difference(series, periods=1):
    """Difference, then drop the NaNs the shift introduces."""
    # TODO
    raise NotImplementedError


def make_stationary(series, max_diff=2, significance=0.05):
    """Difference until ADF calls the series stationary.

    Returns (series, n_differences). Already stationary => unchanged, 0.
    Stop at `max_diff` and return what you have even if ADF still refuses.
    """
    # TODO
    raise NotImplementedError

In [ ]:
walk_stationary, n_diffs = make_stationary(demo_random_walk)
print(f"random walk needed {n_diffs} difference(s); "
      f"p-value now {adf_test(walk_stationary)['p_value']:.2e}")

series_stationary, series_diffs = make_stationary(series)
print(f"temperatures needed {series_diffs} difference(s)")

**Question.** The series has a strong yearly cycle, yet ADF calls it
stationary with zero differences. Is a seasonal series stationary? What is ADF
sensitive to, and what does it ignore?

## 2. Autocorrelation analysis

- **ACF** at lag k — correlation of `y[t]` with `y[t-k]`, everything in between
  included.
- **PACF** at lag k — the same, with the intermediate lags partialled out.

Rule of thumb for Practice 4: PACF cuts off after lag p => AR(p); ACF cuts off
after lag q => MA(q).

In [ ]:
def acf_values(series, nlags):
    """Autocorrelations, lags 0..nlags, as a 1-D numpy array."""
    # TODO
    raise NotImplementedError


def pacf_values(series, nlags):
    """Partial autocorrelations, lags 0..nlags, as a 1-D numpy array."""
    # TODO
    raise NotImplementedError

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 8))
plot_acf(series, lags=40, ax=axes[0, 0], title="ACF — temperatures")
plot_pacf(series, lags=40, ax=axes[0, 1], title="PACF — temperatures")
plot_acf(demo_random_walk, lags=40, ax=axes[1, 0], title="ACF — random walk")
plot_pacf(demo_random_walk, lags=40, ax=axes[1, 1], title="PACF — random walk")
plt.tight_layout();

In [ ]:
pacf_temps = pacf_values(series, 10)
print("PACF lags 1..10:", np.round(pacf_temps[1:], 3))
print("\nlags whose |PACF| exceeds the 95% band:",
      [lag for lag in range(1, 11)
       if abs(pacf_temps[lag]) > 1.96 / np.sqrt(len(series))])

**Question.** The random walk's ACF decays slowly while its PACF cuts
off after lag 1. What does that pattern mean, and how does one difference change
it?

## 3. Forecasting with an AR model

AR(p) regresses `y[t]` on its own previous p values. Pick p from the PACF plot
above, then forecast the held-out year.

In [ ]:
def ar_forecast(train, index, lags):
    """Fit AutoReg(lags) on `train`, forecast over `index`."""
    # TODO: AutoReg(..., lags=lags, old_names=False).fit(), then .forecast(n).
    # Fit on `train` only.
    raise NotImplementedError

In [ ]:
train, test = series.iloc[:-365], series.iloc[-365:]
forecast = ar_forecast(train, test.index, lags=7)

plt.figure(figsize=(20, 5))
plt.plot(train.iloc[-730:], label="train")
plt.plot(test, label="test")
plt.plot(forecast, label="AR(7) forecast")
plt.legend()
plt.show()

error = np.sqrt(np.mean((test.to_numpy() - forecast.to_numpy()) ** 2))
print(f"AR(7) RMSE: {error:.4f}")

**Question.** The forecast flattens after a few steps. Why does a pure
AR model converge to the series mean, and what does that mean for a 365-day
horizon?

## 4. Singular Spectrum Analysis (SSA)

Decomposes a series without assuming a model. Three steps:

1. **Embed** — stack lagged windows of length L into a trajectory matrix.
2. **Decompose** — SVD; each singular triple gives one elementary matrix.
3. **Reconstruct** — sum the elementary matrices you want, then average the
   anti-diagonals back into a series (*diagonal averaging*).

The SVD is exact, so summing **all** components must return the original series.
The tests check that, and so should you while debugging.

In [ ]:
def trajectory_matrix(series, window_length):
    """Hankel matrix of lagged windows — step 1 of SSA.

    N points, window L -> an (L, K) matrix, K = N - L + 1, column k being
    series[k : k + L].
    """
    values = np.asarray(series, dtype=float)
    n_points = len(values)
    n_columns = n_points - window_length + 1
    if n_columns < 1:
        raise ValueError("window_length must not exceed len(series)")
    return np.column_stack([values[k:k + window_length] for k in range(n_columns)])

In [ ]:
def diagonal_average(matrix):
    """Average the anti-diagonals of an (L, K) matrix back into a series.

    Element i is the mean of every matrix[r, c] with r + c == i.
    Result length is L + K - 1.
    """
    # TODO: np.fliplr turns anti-diagonals into real diagonals, then
    # np.diagonal(offset=...) walks them.
    raise NotImplementedError


def ssa_reconstruct(series, window_length, indices):
    """Reconstruct a series from the SSA components in `indices`."""
    # TODO: trajectory matrix -> SVD -> rebuild the elementary matrices for
    # `indices` -> sum -> diagonal_average.
    # All components must give back the original series.
    raise NotImplementedError

In [ ]:
window = 365
full = ssa_reconstruct(series, window, range(min(window, len(series) - window + 1)))
print(f"max |original - full reconstruction| = {float((series - full).abs().max()):.2e}")

trend_component = ssa_reconstruct(series, window, [0])
seasonal_component = ssa_reconstruct(series, window, [1, 2])

plt.figure(figsize=(20, 5))
plt.plot(series, alpha=0.3, label="original")
plt.plot(trend_component, label="component 0 (trend)")
plt.plot(trend_component + seasonal_component, label="components 0-2")
plt.legend();

Pick L between N/4 and N/3, plot the singular values, and decide how
many components carry signal rather than noise.

**Question.** Components 1 and 2 usually arrive as a pair with nearly equal
singular values. What structure produces a *pair*, and why can one component
alone not represent it?

In [ ]:
matrix = trajectory_matrix(series, window)
singular_values = np.linalg.svd(matrix, compute_uv=False)

plt.figure(figsize=(12, 4))
plt.semilogy(singular_values[:30], "o-")
plt.xlabel("component")
plt.ylabel("singular value (log)")
plt.title("SSA spectrum")
plt.grid(True);

In [ ]:
# your code here — free exploration, not graded